# 00 Environment and stack map

## Learning objectives

- distinguish offline learning readiness from cloud authorization;
- locate MLflow, Databricks, Microsoft Foundry, and Azure AI Search in one
  application lifecycle;
- verify that provider names and endpoints live in configuration, not notebooks;
- keep every connected call behind an explicit opt-in.

This course uses an original synthetic incident-response scenario. The default
path makes no network request, installs nothing, and uses no credential. It
keeps the useful workshop rhythm of observable failure, focused TODO, check,
and reference solution while applying this repository's runtime contracts.


## Architecture in one sentence

Databricks is the governed lifecycle and serving plane, MLflow 3 records traces
and release evidence, Microsoft Foundry or Databricks supplies configured model
endpoints, and Azure AI Search or Databricks AI Search supplies a configured
retriever behind the same logical name.

The index, endpoint, identity, roles, and permissions are externally
provisioned platform resources. A notebook can verify or use them; it must not
create them or broaden permissions.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()


## Interpret the preflight

`connected_ready=False` is expected in a fresh clone. It says the tracked
configuration still contains placeholders; it does not say your Azure identity
is invalid. Configuration readiness, authentication, authorization, resource
existence, and provider capability are separate checkpoints.

The summary deliberately omits raw provider configuration. Serializing the
whole settings object is unsafe in a real app because runtime configuration can
contain secret references and the App environment contains a live OAuth secret.


## Ownership map

| Concern | Owner in this stack | Evidence |
|---|---|---|
| Resource names and identity | config + platform process | safe preflight |
| Retrieval and documents | `aai-core` provider adapter | `RETRIEVER` span |
| Routing and action policy | packaged application code | tests + trace |
| Comparison and decision | MLflow + `aai-core` gate | run + gate result |
| HTTP serving | MLflow Agent Server on Databricks Apps | bundle/app deployment |

Notebooks explain and exercise these boundaries. Production logic graduates to
`src/` through the `rag-app` or `agent-app` template.


In [ ]:
# YOUR TURN — TODO: assign one accountable owner to each lifecycle concern.
learner_owner = {
    "retrieval_quality": "application_team",
    "search_service_and_roles": "platform_team",
    "release_gate": "application_team",
    "production_action_approval": "incident_commander",
}
learner_owner


In [ ]:
# CHECK YOUR WORK
required = {
    "retrieval_quality",
    "search_service_and_roles",
    "release_gate",
    "production_action_approval",
}
assert set(learner_owner) == required
assert learner_owner["search_service_and_roles"] == "platform_team"
assert learner_owner["production_action_approval"] != "model"
"Ownership boundary is explicit."


In [ ]:
# Reference solution
reference_owner = {
    "retrieval_quality": "application_team",
    "search_service_and_roles": "platform_team",
    "release_gate": "application_team",
    "production_action_approval": "incident_commander",
}
assert learner_owner == reference_owner


## Optional connected readiness

Copy the Azure Search example to `config/aai-platform.yml`, replace only
externally provisioned identifiers, and authenticate with Azure CLI. Do not add
an API key, PAT, client secret, or raw Key Vault value. The Databricks Search
example preserves the same logical model, embedding, and retriever names so the
application code remains unchanged.


In [ ]:
RUN_CONNECTED = False
connected = None
if RUN_CONNECTED:
    connected = session.connected_components(allow_network=True)
    assert set(connected) == {"model", "embedding", "retriever"}
connected


## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why can configuration readiness pass while authorization still fails?
2. Which layer is allowed to create a search index or role assignment?
3. Why does application code use operations-knowledge instead of an index name?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You now have a credential-free course kernel, a safe configuration summary,
and a responsibility map. No provider call, permission change, or resource
creation occurred. Lesson 01 adds trusted access scope, routing, and a human
checkpoint before operational side effects.
